# 02 - Limpieza / ETL y construcción de indicadores de riesgo (SERCOP)

**Proyecto de tesis — Detección de anomalías en las compras públicas del Ecuador mediante datos abiertos SERCOP**
**Catálogo UIDE, ítem #42**

## Objetivo de este notebook

Partiendo de `base_modelado_v1.parquet` (salida del notebook 1, 2,753,194 procesos tras excluir las convocatorias que nunca avanzaron de la etapa de planeación), este notebook:

1. Normaliza el campo `tipo_contratacion` a categorías canónicas.
2. Diagnostica y documenta el tratamiento de los nulos estructurales en las variables de concurrencia (`n_oferentes`, periodo de licitación).
3. Consolida el monto de cada proceso (`monto_final`), documentando su fuente.
4. Construye y **valida con datos reales** cuatro indicadores de riesgo:
   - Desviación de precio (`desviacion_precio_log_z`)
   - Fragmentación de contratos (`n_contratos_30d_no_catalogo`)
   - Proveedor único (`proveedor_unico`)
   - Concentración entidad-proveedor (`concentracion_entidad_proveedor`)

**Nota importante para el equipo:** en varios de estos indicadores, el primer intento de cálculo dio un resultado poco creíble o inconsistente (por ejemplo, marcar el 19% o el 56% de la base como "atípica"). En vez de aceptar ese resultado, se diagnosticó la causa con consultas adicionales y se corrigió el método. Esos intentos fallidos y su corrección quedan documentados aquí a propósito — es evidencia de validación metodológica, un punto que la tutora ha pedido explícitamente ("midieron y varias de sus conclusiones corrigen supuestos previos").


## 1. Configuración y carga de la base de modelado

In [ ]:
import duckdb
import os

CARPETA_RESULTADOS = r"C:\SERCOP_FINAL\03_resultados"
RUTA_BASE_MODELADO = os.path.join(CARPETA_RESULTADOS, "base_modelado_v1.parquet").replace("\\", "/")

con = duckdb.connect(database=':memory:')

print(f"Base de modelado: {RUTA_BASE_MODELADO}")
n_filas = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_BASE_MODELADO}')").fetchone()[0]
print(f"Filas: {n_filas:,}")

**Resultado:** `2,753,194` filas — coincide con la salida del notebook 1 (consolidado 2015-2026 menos los 32,161 procesos que se quedaron solo en `planning`).

## 2. Normalización de `tipo_contratacion`

El campo crudo (`procurementMethodDetails` de OCDS) trae pegado el nombre completo del convenio marco al tipo real de contratación (ej. *"Catálogo electrónico - Mejor oferta en el convenio SERCOP-SELPROV-026-2017, SUMINISTROS DE OFICINA..."*). Se recorta todo lo que sigue a `"en el convenio"` para obtener la categoría real.

In [ ]:
# --- Antes de normalizar: cuántas categorías distintas hay en crudo ---
n_categorias_crudas = con.execute(f"""
    SELECT COUNT(DISTINCT tipo_contratacion) AS n
    FROM read_parquet('{RUTA_BASE_MODELADO}')
""").fetchone()[0]
print(f"Categorías distintas de tipo_contratacion SIN normalizar: {n_categorias_crudas:,}")

In [ ]:
# --- Normalización: todo lo que sigue a "en el convenio" se recorta ---
query_normalizado = f"""
    SELECT
        *,
        CASE
            WHEN tipo_contratacion LIKE '%en el convenio%'
                THEN TRIM(SPLIT_PART(tipo_contratacion, 'en el convenio', 1))
            ELSE TRIM(tipo_contratacion)
        END AS tipo_contratacion_canonico
    FROM read_parquet('{RUTA_BASE_MODELADO}')
"""

n_categorias_canonicas = con.execute(f"""
    SELECT COUNT(DISTINCT tipo_contratacion_canonico) AS n
    FROM ({query_normalizado})
""").fetchone()[0]
print(f"Categorías distintas de tipo_contratacion_canonico DESPUÉS de normalizar: {n_categorias_canonicas:,}")

distribucion_canonica = con.execute(f"""
    SELECT tipo_contratacion_canonico, COUNT(*) AS n
    FROM ({query_normalizado})
    GROUP BY 1
    ORDER BY n DESC
""").fetchdf()
print("\nDistribución completa de tipo_contratacion_canonico:")
print(distribucion_canonica.to_string(index=False))

**Resultado:** de **1,644 categorías crudas a 27 categorías canónicas**. Las dos más frecuentes son `Catálogo electrónico - Mejor oferta` (1,411,761 procesos, 51.3%) y `Catálogo electrónico - Compra directa` (850,749, 30.9%) — juntas son el 82% de toda la base.

In [ ]:
# --- Guardar tipo_contratacion_canonico como columna definitiva ---
RUTA_V2 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v2.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_normalizado})
    TO '{RUTA_V2}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v2 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V2}')").fetchone()[0]
assert n_v2 == n_filas
print(f"base_modelado_v2.parquet: {n_v2:,} filas — OK")

## 3. ¿Por qué `n_oferentes` tiene tantos nulos? Diagnóstico antes de decidir

La tutora pidió explícitamente revisar esto: *"dividan la base según si esas variables están presentes o ausentes"*. Se cruza el % de nulos de `n_oferentes` y `fecha_inicio_tender` contra `tipo_contratacion_canonico`, en vez de asumir que el nulo es un problema de relleno.

In [ ]:
nulos_por_tipo = con.execute(f"""
    SELECT
        tipo_contratacion_canonico,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(CASE WHEN n_oferentes IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_n_oferentes_nulo,
        ROUND(100.0 * SUM(CASE WHEN fecha_inicio_tender IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_fecha_inicio_nulo
    FROM read_parquet('{RUTA_V2}')
    GROUP BY 1
    ORDER BY n DESC
""").fetchdf()
print(nulos_por_tipo.to_string(index=False))

**Resultado clave:** el nulo es **binario por categoría** — cada tipo de contratación tiene 0% o 100% de nulos en `n_oferentes` (con una única excepción, `Subasta Inversa Electrónica`, al 1.2%). Esto demuestra que el campo **no existe** para compra directa/catálogo electrónico (no hay etapa competitiva que reportar), en vez de haberse perdido por error.

**Dato curioso a documentar:** `Contratacion directa` (sin tilde, 31,538 casos, 0% nulo) y `Catálogo electrónico - Compra directa` (850,749 casos, 100% nulo) son categorías distintas pese al nombre parecido — solo la primera sí registra periodo competitivo. Por eso la bandera de "tiene periodo competitivo" se construye desde el dato real, no desde el texto de la categoría.

In [ ]:
# --- Construir bandera estructural: tiene_periodo_competitivo (desde el dato, no desde el nombre de la categoría) ---
query_con_bandera = f"""
    SELECT
        *,
        (fecha_inicio_tender IS NOT NULL) AS tiene_periodo_competitivo
    FROM read_parquet('{RUTA_V2}')
"""

resumen_bandera = con.execute(f"""
    SELECT tiene_periodo_competitivo, COUNT(*) AS n, ROUND(100.0 * COUNT(*) / {n_v2}, 1) AS pct_del_total
    FROM ({query_con_bandera})
    GROUP BY 1
""").fetchdf()
print(resumen_bandera.to_string(index=False))

**Resultado:** `TRUE` (sí hubo periodo competitivo) = **467,323 filas (17.0%)**; `FALSE` = 2,285,871 (83.0%).

**Decisión de modelado (documentar en el capítulo de metodología):** no se imputa el 83% sin periodo competitivo, ni se descarta esa porción de la base (perder el 83% eliminaría justo los métodos — compra directa y catálogo — más asociados a fraccionamiento y colusión en la literatura). En vez de eso, se definen dos familias de variables:

- **Universales** (se calculan para el 100% de la base): desviación de precio, fragmentación, concentración entidad-proveedor.
- **De concurrencia** (solo se calculan y se usan como variable de entrada al modelo sobre el subconjunto `tiene_periodo_competitivo = TRUE`): proveedor único. Para el resto de la base, esta columna queda `NULL` explícito — no se imputa, no se usa como feature en ese subconjunto.

In [ ]:
RUTA_V3 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v3.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_con_bandera})
    TO '{RUTA_V3}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v3 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V3}')").fetchone()[0]
assert n_v3 == n_v2
print(f"base_modelado_v3.parquet: {n_v3:,} filas — OK")

## 4. Consolidar el monto del proceso (`monto_final`)

Hay dos fuentes de monto en el estándar OCDS: `tender.value.amount` (presupuesto referencial, antes de competir) y `awards[].value.amount` (monto realmente adjudicado). Antes de decidir cómo combinarlos, se revisa cuántos procesos tienen cada combinación.

In [ ]:
combinaciones_monto = con.execute(f"""
    SELECT
        (monto_tender IS NOT NULL) AS tiene_monto_tender,
        (monto_award IS NOT NULL) AS tiene_monto_award,
        COUNT(*) AS n
    FROM read_parquet('{RUTA_V3}')
    GROUP BY 1, 2
    ORDER BY n DESC
""").fetchdf()
print(combinaciones_monto.to_string(index=False))

**Resultado:** 2,515,863 procesos tienen ambos montos; 216,469 solo tienen `award`; 4,055 solo tienen `tender`; **16,807 no tienen ninguno de los dos**.

**Regla de consolidación:** `monto_final = COALESCE(monto_award, monto_tender)` — se prioriza el monto adjudicado porque es el valor real de la transacción, más preciso que el presupuesto referencial para detectar sobreprecios.

In [ ]:
query_con_monto_final = f"""
    SELECT
        *,
        COALESCE(monto_award, monto_tender) AS monto_final,
        CASE
            WHEN monto_award IS NOT NULL THEN 'award'
            WHEN monto_tender IS NOT NULL THEN 'tender'
            ELSE 'sin_dato'
        END AS fuente_monto
    FROM read_parquet('{RUTA_V3}')
"""

resumen_fuente = con.execute(f"""
    SELECT fuente_monto, COUNT(*) AS n, ROUND(100.0*COUNT(*)/{n_v3}, 2) AS pct
    FROM ({query_con_monto_final})
    GROUP BY 1
    ORDER BY n DESC
""").fetchdf()
print(resumen_fuente.to_string(index=False))

**Resultado:** 99.24% desde `award`, 0.15% desde `tender`, **0.61% (16,807) sin dato en ninguna fuente**.

In [ ]:
RUTA_V4 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v4.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_con_monto_final})
    TO '{RUTA_V4}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v4 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V4}')").fetchone()[0]
assert n_v4 == n_v3
print(f"base_modelado_v4.parquet: {n_v4:,} filas — OK")

## 5. Separar los procesos sin ningún monto

Sin `monto_final`, no se puede calcular ningún indicador de precio. Se separan a un archivo aparte (no se descartan del proyecto, quedan disponibles para otro tipo de análisis) y la base de modelado continúa sin ellos.

In [ ]:
RUTA_SIN_MONTO = os.path.join(CARPETA_RESULTADOS, "procesos_sin_monto.parquet").replace("\\", "/")
RUTA_V5 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v5.parquet").replace("\\", "/")

con.execute(f"""
    COPY (SELECT * FROM read_parquet('{RUTA_V4}') WHERE fuente_monto = 'sin_dato')
    TO '{RUTA_SIN_MONTO}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.execute(f"""
    COPY (SELECT * FROM read_parquet('{RUTA_V4}') WHERE fuente_monto != 'sin_dato')
    TO '{RUTA_V5}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_sin_monto = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_SIN_MONTO}')").fetchone()[0]
n_v5 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V5}')").fetchone()[0]

print(f"Sin monto (excluidos de indicadores de precio): {n_sin_monto:,}")
print(f"Base de modelado v5 (con monto):                {n_v5:,}")
assert n_sin_monto + n_v5 == n_v4
print("OK: reconcilia con base_modelado_v4.")

**Resultado:** 16,807 excluidos; **base de modelado con monto = 2,736,387 filas.** Esta es la base sobre la que se construyen todos los indicadores de riesgo a partir de aquí.

## 6. Indicador de riesgo 1 — Desviación de precio

### Intento 1 (fallido): z-score robusto sobre el monto crudo

Se calcula la mediana y el MAD (desviación absoluta mediana — más robusto que la media/desviación estándar para datos monetarios sesgados) de `monto_final`, agrupado por `tipo_contratacion_canonico`, y el z-score robusto: `0.6745 * (x - mediana) / MAD`.

In [ ]:
# Paso A: mediana simple por tipo
mediana_simple = con.execute(f"""
    SELECT tipo_contratacion_canonico, COUNT(*) AS n, MEDIAN(monto_final) AS mediana_monto
    FROM read_parquet('{RUTA_V5}')
    GROUP BY 1
""").fetchdf()

# Paso B: MAD por tipo (DuckDB no permite anidar MEDIAN(...) OVER(...) dentro de otro agregado,
# así que se calcula en dos pasos con un JOIN)
query_medianas = f"""
    SELECT tipo_contratacion_canonico, MEDIAN(monto_final) AS mediana_monto
    FROM read_parquet('{RUTA_V5}')
    GROUP BY 1
"""
query_mad = f"""
    WITH medianas AS ({query_medianas}),
    con_mediana AS (
        SELECT b.*, m.mediana_monto
        FROM read_parquet('{RUTA_V5}') b
        JOIN medianas m USING (tipo_contratacion_canonico)
    )
    SELECT tipo_contratacion_canonico, MEDIAN(mediana_monto) AS mediana_monto,
           MEDIAN(ABS(monto_final - mediana_monto)) AS mad_monto
    FROM con_mediana
    GROUP BY 1
"""
mad_por_tipo = con.execute(query_mad).fetchdf()

# Paso C: z-score robusto sobre monto crudo
query_con_desviacion = f"""
    WITH stats AS ({query_mad}),
    con_stats AS (
        SELECT b.*, s.mediana_monto AS mediana_monto_tipo, s.mad_monto AS mad_monto_tipo
        FROM read_parquet('{RUTA_V5}') b
        JOIN stats s USING (tipo_contratacion_canonico)
    )
    SELECT *, 0.6745 * (monto_final - mediana_monto_tipo) / NULLIF(mad_monto_tipo, 0) AS desviacion_precio_z
    FROM con_stats
"""

diag_z = con.execute(f"""
    SELECT COUNT(*) AS n,
           SUM(CASE WHEN desviacion_precio_z IS NULL THEN 1 ELSE 0 END) AS n_nulo,
           MIN(desviacion_precio_z) AS min_z, MAX(desviacion_precio_z) AS max_z,
           SUM(CASE WHEN ABS(desviacion_precio_z) > 3.5 THEN 1 ELSE 0 END) AS n_atipicos_umbral_3_5
    FROM ({query_con_desviacion})
""").fetchdf()
print(diag_z.to_string(index=False))

**Resultado del intento 1: `531,781` procesos marcados como atípicos — 19.4% de la base.** Esto NO es un resultado útil: un indicador que marca 1 de cada 5 compras públicas como "atípica" no discrimina nada. `max_z = 93,424` confirma el problema — un valor extremo domina toda la escala.

**Diagnóstico de la causa:** los montos son datos monetarios con distribución muy sesgada a la derecha (muchas compras pequeñas + pocos contratos gigantes dentro del mismo tipo de contratación). Calcular el z-score sobre el monto **crudo** hace que esa cola larga domine el cálculo.

### Intento 2 (corregido): z-score robusto sobre el LOGARITMO del monto

Se aplica logaritmo natural antes de calcular mediana/MAD — práctica estándar para variables monetarias con cola larga. Antes de eso, se revisan los montos en cero (log indefinido en 0).

In [ ]:
diag_monto = con.execute(f"""
    SELECT MIN(monto_final) AS min_monto,
           SUM(CASE WHEN monto_final = 0 THEN 1 ELSE 0 END) AS n_monto_cero,
           SUM(CASE WHEN monto_final < 0 THEN 1 ELSE 0 END) AS n_monto_negativo
    FROM read_parquet('{RUTA_V5}')
""").fetchdf()
print(diag_monto.to_string(index=False))

**Resultado:** 16 procesos con `monto_final = 0` (nada negativo). Un contrato público en $0 es en sí mismo un caso a documentar, no un dato a transformar sin más — se marcan aparte (`monto_cero = TRUE`) y no entran al cálculo de logaritmo.

In [ ]:
query_base_log = f"""
    SELECT
        *,
        COALESCE(tipo_contratacion_canonico, 'SIN_TIPO_CONTRATACION') AS tipo_contratacion_grp,
        (monto_final = 0) AS monto_cero,
        CASE WHEN monto_final > 0 THEN LN(monto_final) ELSE NULL END AS log_monto
    FROM read_parquet('{RUTA_V5}')
"""

query_medianas_log = f"""
    SELECT tipo_contratacion_grp, MEDIAN(log_monto) AS mediana_log
    FROM ({query_base_log})
    WHERE log_monto IS NOT NULL
    GROUP BY 1
"""

query_mad_log = f"""
    WITH medianas AS ({query_medianas_log}),
    con_mediana AS (
        SELECT b.*, m.mediana_log
        FROM ({query_base_log}) b
        JOIN medianas m USING (tipo_contratacion_grp)
    )
    SELECT tipo_contratacion_grp, MEDIAN(mediana_log) AS mediana_log,
           MEDIAN(ABS(log_monto - mediana_log)) AS mad_log
    FROM con_mediana
    WHERE log_monto IS NOT NULL
    GROUP BY 1
"""

query_final_z = f"""
    WITH stats AS ({query_mad_log}),
    con_stats AS (
        SELECT b.*, s.mediana_log, s.mad_log
        FROM ({query_base_log}) b
        JOIN stats s USING (tipo_contratacion_grp)
    )
    SELECT *, 0.6745 * (log_monto - mediana_log) / NULLIF(mad_log, 0) AS desviacion_precio_log_z
    FROM con_stats
"""

n_check2 = con.execute(f"SELECT COUNT(*) FROM ({query_final_z})").fetchone()[0]
print(f"Filas resultantes: {n_check2:,}  (debería ser 2,736,387)")

diag_z2 = con.execute(f"""
    SELECT COUNT(*) AS n,
           SUM(CASE WHEN monto_cero THEN 1 ELSE 0 END) AS n_monto_cero,
           SUM(CASE WHEN desviacion_precio_log_z IS NULL AND NOT monto_cero THEN 1 ELSE 0 END) AS n_nulo_inesperado,
           MIN(desviacion_precio_log_z) AS min_z, MAX(desviacion_precio_log_z) AS max_z,
           SUM(CASE WHEN ABS(desviacion_precio_log_z) > 3.5 THEN 1 ELSE 0 END) AS n_atipicos_umbral_3_5
    FROM ({query_final_z})
""").fetchdf()
print(diag_z2.to_string(index=False))

**Resultado del intento 2: `2,736,387` filas (correcto), y solo `6,926` atípicos (0.25%)** — resultado mucho más razonable y defendible como candidatos reales de revisión (umbral estándar |z|>3.5, Iglewicz & Hoaglin 1993).

El único nulo inesperado corresponde al único proceso con `tipo_contratacion_canonico = NULL` (grupo de un solo elemento; su MAD es 0 por definición, no hay forma de medir dispersión con un dato). Caso aislado, documentado, no es un error.

In [ ]:
RUTA_V6 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v6.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_final_z})
    TO '{RUTA_V6}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v6 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V6}')").fetchone()[0]
assert n_v6 == n_check2
print(f"base_modelado_v6.parquet: {n_v6:,} filas — OK")

## 7. Indicador de riesgo 2 — Fragmentación de contratos

Fragmentación: dividir una compra grande en varias chicas para evitar el proceso competitivo obligatorio que activa un monto mayor (red flag documentado en Fazekas & Kocsis 2020 y en la guía "Red Flags for Integrity" de la Open Contracting Partnership). Se mide como: mismo par entidad-proveedor con múltiples contratos en una ventana móvil de 30 días.

**Limitación documentada:** no se extrajo aún `dateSigned` (fecha de firma del contrato, anidada en `contracts[]`); se usa `fecha_release` como proxy.

### Intento 1 (fallido): sobre toda la base

In [ ]:
query_fragmentacion = f"""
    SELECT
        *,
        CASE WHEN entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL THEN
            COUNT(*) OVER (
                PARTITION BY entidad_ruc, proveedor_ruc
                ORDER BY fecha_release
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            )
        ELSE NULL END AS n_contratos_30d_par,
        CASE WHEN entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL THEN
            SUM(monto_final) OVER (
                PARTITION BY entidad_ruc, proveedor_ruc
                ORDER BY fecha_release
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            )
        ELSE NULL END AS monto_acumulado_30d_par
    FROM read_parquet('{RUTA_V6}')
"""

resumen_frag = con.execute(f"""
    SELECT
        COUNT(*) AS n_con_par_ruc,
        SUM(CASE WHEN n_contratos_30d_par >= 3 THEN 1 ELSE 0 END) AS n_posible_fragmentacion_3mas,
        SUM(CASE WHEN n_contratos_30d_par >= 5 THEN 1 ELSE 0 END) AS n_posible_fragmentacion_5mas
    FROM ({query_fragmentacion})
    WHERE n_contratos_30d_par IS NOT NULL
""").fetchdf()
print(resumen_frag.to_string(index=False))

**Resultado del intento 1: 1,546,055 de 2,732,311 (56.6%) marcados con 3+ contratos en 30 días.** Otro indicador inútil de tan amplio.

In [ ]:
# --- Diagnóstico de la causa ---
diag_por_tipo = con.execute(f"""
    SELECT tipo_contratacion_grp, COUNT(*) AS n
    FROM ({query_fragmentacion})
    WHERE n_contratos_30d_par >= 3
    GROUP BY 1
    ORDER BY n DESC
    LIMIT 10
""").fetchdf()
print(diag_por_tipo.to_string(index=False))

**Causa encontrada: 98.8% de esos casos son catálogo electrónico** (`Catálogo electrónico - Mejor oferta`: 1,161,182; `Compra directa`: 366,096). Ahí la recompra frecuente al mismo proveedor es el funcionamiento normal del convenio marco (precios pre-negociados, sin competencia por proceso) — **no es fragmentación**, es la forma en que opera el catálogo.

### Intento 2 (corregido): excluir catálogo electrónico

La fragmentación solo tiene sentido donde existen umbrales de monto que activan procesos más rigurosos (compra directa fuera de catálogo, menor cuantía, etc.). Para catálogo electrónico, el indicador queda `NULL` explícito — no aplica el concepto, no se calcula un 0 falso.

In [ ]:
query_frag_v2 = f"""
    WITH base_no_catalogo AS (
        SELECT *
        FROM ({query_fragmentacion})
        WHERE tipo_contratacion_grp NOT LIKE 'Catálogo electrónico%'
          AND entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL
    ),
    con_ventana AS (
        SELECT
            ocid,
            COUNT(*) OVER (
                PARTITION BY entidad_ruc, proveedor_ruc
                ORDER BY fecha_release
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            ) AS n_contratos_30d_no_catalogo,
            SUM(monto_final) OVER (
                PARTITION BY entidad_ruc, proveedor_ruc
                ORDER BY fecha_release
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            ) AS monto_acumulado_30d_no_catalogo
        FROM base_no_catalogo
    )
    SELECT
        b.* EXCLUDE (n_contratos_30d_par, monto_acumulado_30d_par),
        v.n_contratos_30d_no_catalogo,
        v.monto_acumulado_30d_no_catalogo
    FROM ({query_fragmentacion}) b
    LEFT JOIN con_ventana v USING (ocid)
"""

resumen_frag_v2 = con.execute(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN n_contratos_30d_no_catalogo IS NOT NULL THEN 1 ELSE 0 END) AS n_elegibles,
        SUM(CASE WHEN n_contratos_30d_no_catalogo >= 3 THEN 1 ELSE 0 END) AS n_posible_fragmentacion_3mas,
        SUM(CASE WHEN n_contratos_30d_no_catalogo >= 5 THEN 1 ELSE 0 END) AS n_posible_fragmentacion_5mas
    FROM ({query_frag_v2})
""").fetchdf()
print(resumen_frag_v2.to_string(index=False))

**Resultado del intento 2:** de 446,876 procesos elegibles (no-catálogo, con RUC de ambas partes), **11,321 (2.5%) tienen 3+ contratos en 30 días con el mismo par**, y **1,834 (0.4%) tienen 5+** — indicador con capacidad discriminativa real.

In [ ]:
RUTA_V7 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v7.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_frag_v2})
    TO '{RUTA_V7}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v7 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V7}')").fetchone()[0]
print(f"base_modelado_v7.parquet: {n_v7:,} filas — OK")

## 8. Indicador de riesgo 3 — Proveedor único

Se calcula `n_oferentes = 1` **solo dentro del subconjunto `tiene_periodo_competitivo = TRUE`** (467,323 filas de la sección 3) — corrigiendo así el error que señaló la tutora: el código anterior aplicaba este indicador sin restringirlo al tipo de procedimiento correcto.

In [ ]:
resumen_n_oferentes = con.execute(f"""
    SELECT
        tiene_periodo_competitivo,
        COUNT(*) AS n,
        SUM(CASE WHEN n_oferentes IS NULL THEN 1 ELSE 0 END) AS n_nulo,
        MIN(n_oferentes) AS min_oferentes,
        MAX(n_oferentes) AS max_oferentes,
        SUM(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END) AS n_proveedor_unico
    FROM read_parquet('{RUTA_V7}')
    GROUP BY 1
""").fetchdf()
print(resumen_n_oferentes.to_string(index=False))

**Resultado clave para la tesis:** dentro del subconjunto competitivo (450,931 procesos tras las exclusiones previas), **211,749 (46.96%) tuvieron un solo oferente**. Casi la mitad de los procesos formalmente "competitivos" no tuvieron competencia real en la práctica — un hallazgo sustantivo, no solo una limpieza de datos.

(18 casos con `tiene_periodo_competitivo = TRUE` pero `n_oferentes` nulo — margen mínimo, documentado.)

In [ ]:
query_proveedor_unico = f"""
    SELECT
        *,
        CASE
            WHEN tiene_periodo_competitivo THEN (n_oferentes = 1)
            ELSE NULL
        END AS proveedor_unico
    FROM read_parquet('{RUTA_V7}')
"""

RUTA_V8 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v8.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_proveedor_unico})
    TO '{RUTA_V8}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v8 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V8}')").fetchone()[0]
print(f"base_modelado_v8.parquet: {n_v8:,} filas — OK")

resumen_final = con.execute(f"""
    SELECT proveedor_unico, COUNT(*) AS n
    FROM read_parquet('{RUTA_V8}')
    GROUP BY 1
""").fetchdf()
print(resumen_final.to_string(index=False))

**Resultado:** `proveedor_unico = TRUE` en 211,749 filas, `FALSE` en 239,164, `NULL` (no aplica, sin periodo competitivo) en 2,285,474 — coincide exactamente con lo esperado.

## 9. Indicador de riesgo 4 — Concentración entidad-proveedor

Mide qué proporción del gasto histórico total de una entidad (2015-2026) se concentra en un único proveedor. Es un indicador clásico de riesgo de favoritismo/dependencia (Fazekas & Kocsis 2020, "market concentration").

In [ ]:
query_concentracion = f"""
    SELECT
        *,
        SUM(monto_final) OVER (PARTITION BY entidad_ruc, proveedor_ruc) AS monto_total_historico_par,
        SUM(monto_final) OVER (PARTITION BY entidad_ruc) AS monto_total_historico_entidad,
        COUNT(DISTINCT proveedor_ruc) OVER (PARTITION BY entidad_ruc) AS n_proveedores_distintos_entidad
    FROM read_parquet('{RUTA_V8}')
"""

query_concentracion_final = f"""
    SELECT
        *,
        CASE WHEN entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL THEN
            monto_total_historico_par / NULLIF(monto_total_historico_entidad, 0)
        ELSE NULL END AS concentracion_entidad_proveedor
    FROM ({query_concentracion})
"""

diag_concentracion = con.execute(f"""
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN concentracion_entidad_proveedor IS NULL THEN 1 ELSE 0 END) AS n_nulo,
        MEDIAN(n_proveedores_distintos_entidad) AS mediana_proveedores_entidad,
        MAX(n_proveedores_distintos_entidad) AS max_proveedores_entidad,
        MEDIAN(concentracion_entidad_proveedor) AS mediana_concentracion,
        SUM(CASE WHEN concentracion_entidad_proveedor >= 0.5 THEN 1 ELSE 0 END) AS n_concentracion_50mas,
        SUM(CASE WHEN concentracion_entidad_proveedor >= 0.8 THEN 1 ELSE 0 END) AS n_concentracion_80mas
    FROM ({query_concentracion_final})
""").fetchdf()
print(diag_concentracion.to_string(index=False))

**Resultado:** mediana de concentración muy baja (0.22%), consistente con una mediana de 260 proveedores distintos por entidad. Pero **2,214 procesos (0.08%) pertenecen a pares donde el proveedor concentra el 80%+ del gasto histórico de esa entidad** — señal fuerte y poco frecuente, el perfil típico de un buen indicador de riesgo (discrimina, no satura la base).

In [ ]:
RUTA_V9 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v9.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_concentracion_final})
    TO '{RUTA_V9}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v9 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V9}')").fetchone()[0]
print(f"base_modelado_v9.parquet: {n_v9:,} filas — OK")

cols_v9 = con.execute(f"SELECT * FROM read_parquet('{RUTA_V9}') LIMIT 0").df().columns.tolist()
print(f"\nTotal de columnas: {len(cols_v9)}")
print(cols_v9)

**Resultado:** `base_modelado_v9.parquet`, **2,736,387 filas, 41 columnas** — la base final de este notebook.

## 10. Resumen y notas para el equipo

### Archivos producidos (en `C:\SERCOP_FINAL\03_resultados\`)

- `procesos_solo_planeacion.parquet` — 32,161 filas, procesos que nunca avanzaron de la etapa de planeación (no son compras). Se observó, revisando casos individuales en el portal SERCOP, que la causa suele ser una mala especificación del requerimiento en la justificación inicial, que impide que el proceso avance a licitación.
- `procesos_sin_monto.parquet` — 16,807 filas sin monto en ninguna fuente OCDS.
- **`base_modelado_v9.parquet`** — **2,736,387 filas, 41 columnas** — base final con los 4 indicadores de riesgo, lista para el notebook de modelado.

### Indicadores de riesgo construidos

| Indicador | Cobertura | Casos marcados |
|---|---|---|
| `desviacion_precio_log_z` | 100% de la base (excepto `monto_cero`) | 6,926 (0.25%) con &#124;z&#124;>3.5 |
| `monto_cero` | 100% de la base | 16 procesos en $0 |
| `n_contratos_30d_no_catalogo` | Solo no-catálogo (446,876 elegibles) | 11,321 (2.5%) con 3+ contratos/30 días |
| `proveedor_unico` | Solo `tiene_periodo_competitivo=TRUE` (450,931) | 211,749 (46.96%) |
| `concentracion_entidad_proveedor` | 100% de la base (con RUC en ambas partes) | 2,214 (0.08%) con concentración ≥80% |

### Próximos pasos (notebook 3)

Cruce de `entidad_ruc` / `proveedor_ruc` contra el catastro de RUC del SRI (antigüedad del proveedor al momento de la oferta) y contra el listado de "empresas fantasmas", para construir variables de validación externa — el punto que sigue pendiente de la pregunta central de la tutora: *"contra qué van a validar que una anomalía detectada es realmente una anomalía"*.
